# `structured-pruning` vs `baseline-test` (misma familia de experimentos)

## Qué pipeline y qué modelo usa cada run

| Pieza | Detalle |
|-------|---------|
| **CLI** | `python -m pia.cli.structured_ticket` → `iterative_structured_magnitude_pruning` en `pia.pruning.structured_ticket`. |
| **Poda** | **Poda estructurada por ancho** en cadena: tras cada fase se llama a `narrow_chain_cnn_by_fraction` (puntuación L1 acoplada conv1 ↔ conv2 ↔ bloques de `fc`) y el `state_dict` **cambia de forma** (menos canales reales). No es IMP de ResNet con máscaras element-wise sobre la misma topología. |
| **Red** | `NarrowableChainCnn` (`pia.models.narrowable_chain_cnn`): dos convoluciones 3×3, ReLU, cabeza lineal sobre CIFAR-10 (32×32). |
| **Pérdida** | `SparseLoss`: CE + λ·L1(pesos) + γ·L1(activaciones) |
| **Índice** | Cada run escribe `structured_index.json` (`schema_version` 2) con `meta.pipeline = structured_width_chain_cnn` y una lista `rounds` por fase. |

## Cómo interpretar los dos nombres de run

- **`baseline-test`**: en el snapshot típico del índice, suele ser la **fase densa al ancho inicial** (`initial_conv_width`, p. ej. 32 canales) con `achieved_sparsity` 0 en la última entrada exportada; sirve como **referencia de precisión/tamaño** antes de (o sin) recortes adicionales registrados en ese JSON.
- **`structured-pruning`**: mismo código y modelo base, pero con **varias rondas** de recorte de ancho; `model_final.pt` corresponde al **último ancho entrenado** (mayor esparcidad estructural, menos parámetros reales).

Ajusta las rutas en la siguiente celda si tus carpetas difieren.


In [1]:
"""Rutas: run ancho reducido vs run baseline (misma familia `structured_ticket`)."""

from __future__ import annotations

import sys
from pathlib import Path


def resolve_repo_root() -> Path:
    """Sube desde notebooks/ hasta la raíz (directorio con src/pia)."""
    p = Path.cwd().resolve()
    if (p / "src" / "pia").is_dir():
        return p
    if (p.parent / "src" / "pia").is_dir():
        return p.parent
    msg = f"No se encontró src/pia. cwd={p}"
    raise FileNotFoundError(msg)


REPO_ROOT = resolve_repo_root()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

DATA_ROOT = str(REPO_ROOT / "data")

BASELINE_TEST_DIR = REPO_ROOT / "runs" / "baseline-test" / "baseline-test"
STRUCTURED_PRUNING_DIR = REPO_ROOT / "runs" / "structured-pruning" / "structured-pruning"

BASELINE_INDEX = BASELINE_TEST_DIR / "structured_index.json"
BASELINE_CKPT = BASELINE_TEST_DIR / "model_final.pt"

STRUCTURED_INDEX = STRUCTURED_PRUNING_DIR / "structured_index.json"
STRUCTURED_CKPT = STRUCTURED_PRUNING_DIR / "model_final.pt"

BATCH_SIZE = 128
MAX_TIMING_BATCHES = 30

for label, p in [
    ("baseline-test index", BASELINE_INDEX),
    ("baseline-test ckpt", BASELINE_CKPT),
    ("structured-pruning index", STRUCTURED_INDEX),
    ("structured-pruning ckpt", STRUCTURED_CKPT),
]:
    print(label + ":", p, "exists" if p.is_file() else "MISSING")


baseline-test index: /Users/carlosvalerio/Documents/Study/Aprendizaje/pia/runs/baseline-test/baseline-test/structured_index.json exists
baseline-test ckpt: /Users/carlosvalerio/Documents/Study/Aprendizaje/pia/runs/baseline-test/baseline-test/model_final.pt exists
structured-pruning index: /Users/carlosvalerio/Documents/Study/Aprendizaje/pia/runs/structured-pruning/structured-pruning/structured_index.json exists
structured-pruning ckpt: /Users/carlosvalerio/Documents/Study/Aprendizaje/pia/runs/structured-pruning/structured-pruning/model_final.pt exists


In [2]:
"""Carga, reconstrucción NarrowableChainCnn, test CIFAR-10, métricas."""

from __future__ import annotations

import json
import math
import statistics
import time
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import display
import torch
from torch import nn

from pia.data.cifar10 import build_cifar10_test_loader
from pia.models.narrowable_chain_cnn import NarrowableChainCnn
from pia.training.metrics import weight_sparsity_ratio


def load_state_dict(path: Path) -> dict[str, torch.Tensor]:
    try:
        raw = torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        raw = torch.load(path, map_location="cpu")
    if not isinstance(raw, dict):
        msg = "Se esperaba un dict (state_dict)."
        raise TypeError(msg)
    return raw


def load_index(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def build_chain_from_state_dict(sd: dict[str, torch.Tensor]) -> NarrowableChainCnn:
    w1 = sd["conv1.weight"]
    width, in_ch, *_ = w1.shape
    fc_w = sd["fc.weight"]
    _n_cls, flat = fc_w.shape
    if flat % width != 0:
        msg = f"fc.in_features={flat} no divisible por width={width}"
        raise ValueError(msg)
    hw = flat // width
    s = int(math.isqrt(hw))
    if s * s != hw:
        msg = f"Mapa no cuadrado: hw={hw}"
        raise ValueError(msg)
    n_cls = int(fc_w.shape[0])
    m = NarrowableChainCnn(in_ch, width, (s, s), n_cls)
    m.load_state_dict(sd)
    return m


def pick_device() -> torch.device:
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def eval_test_accuracy(model: nn.Module, loader: Any, device: torch.device) -> float:
    model.eval()
    model.to(device)
    total_correct = 0
    total_n = 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            pred = model(x).argmax(dim=1)
            total_correct += int((pred == y).sum().item())
            total_n += int(y.shape[0])
    return total_correct / max(1, total_n)


def time_forward_batches(
    model: nn.Module, loader: Any, device: torch.device, *, max_batches: int
) -> tuple[float, float]:
    model.eval()
    model.to(device)
    it = iter(loader)
    x0, _ = next(it)
    x0 = x0.to(device, non_blocking=True)
    with torch.no_grad():
        _ = model(x0)
    if device.type == "cuda":
        torch.cuda.synchronize()
    lat_ms: list[float] = []
    n = 0
    with torch.no_grad():
        for x, _ in loader:
            if n >= max_batches:
                break
            x = x.to(device, non_blocking=True)
            t0 = time.perf_counter()
            _ = model(x)
            if device.type == "cuda":
                torch.cuda.synchronize()
            lat_ms.append((time.perf_counter() - t0) * 1000.0)
            n += 1
    if len(lat_ms) < 2:
        m = float(lat_ms[0]) if lat_ms else 0.0
        return m, 0.0
    return statistics.mean(lat_ms), statistics.stdev(lat_ms)


def model_param_bytes(module: nn.Module) -> tuple[int, int]:
    param_b = sum(p.numel() * p.element_size() for p in module.parameters())
    n_params = sum(p.numel() for p in module.parameters())
    return param_b, n_params


device = pick_device()
print("device:", device)


device: mps


In [3]:
"""Resumen de índices: meta compartida y métricas por ronda."""

baseline_raw = load_index(BASELINE_INDEX)
struct_raw = load_index(STRUCTURED_INDEX)

meta_rows = [
    {"key": k, "baseline-test": baseline_raw["meta"].get(k), "structured-pruning": struct_raw["meta"].get(k)}
    for k in sorted(set(baseline_raw["meta"]) | set(struct_raw["meta"]))
]
display(pd.DataFrame(meta_rows))


def rounds_df(raw: dict[str, Any], label: str) -> pd.DataFrame:
    rows = []
    for r in raw["rounds"]:
        fm = r.get("final_metrics") or {}
        rows.append(
            {
                "run": label,
                "round": r.get("round"),
                "val/acc": fm.get("val/acc"),
                "val/loss": fm.get("val/loss"),
                "achieved_sparsity": r.get("achieved_sparsity"),
                "width": r.get("structured_conv_width"),
                "n_params": r.get("structured_param_count"),
            }
        )
    return pd.DataFrame(rows)


display(rounds_df(baseline_raw, "baseline-test"))
display(rounds_df(struct_raw, "structured-pruning"))


,key,baseline-test,structured-pruning
0,early_stop_patience,2,2
1,early_stop_val_loss_min_delta,0.0,0.0
2,early_stop_val_loss_relative,0.05,0.05
3,initial_conv_width,32,32
4,lr_scheduler_kind,cosine,cosine
5,lr_step_gamma,0.1,0.1
6,lr_step_size,10,10
7,pipeline,structured_width_chain_cnn,structured_width_chain_cnn
8,rewind_mode,none,none
9,weight_l1_aggregation,sum,sum


,run,round,val/acc,val/loss,achieved_sparsity,width,n_params
0,baseline-test,0,0.611328,1.11746,0.0,32,337770


,run,round,val/acc,val/loss,achieved_sparsity,width,n_params
0,structured-pruning,0,0.469922,2.393237,0.09375,29,305322
1,structured-pruning,1,0.613867,1.383461,0.15625,27,283780
2,structured-pruning,2,0.662891,1.222428,0.21875,25,262310
3,structured-pruning,3,0.686719,1.143060,0.28125,23,240912
4,structured-pruning,4,0.697266,1.084117,0.34375,21,219586
5,structured-pruning,5,0.696680,1.056852,0.34375,21,219586


In [4]:
"""Comparación en CIFAR-10 test: precisión, parámetros, sparsity, latencia aproximada."""

test_loader = build_cifar10_test_loader(
    data_root=DATA_ROOT, batch_size=BATCH_SIZE, num_workers=0
)

rows = []
for name, ckpt in [
    ("baseline-test (model_final)", BASELINE_CKPT),
    ("structured-pruning (model_final)", STRUCTURED_CKPT),
]:
    sd = load_state_dict(ckpt)
    m = build_chain_from_state_dict(sd)
    acc = eval_test_accuracy(m, test_loader, device)
    pb, npar = model_param_bytes(m)
    sp = weight_sparsity_ratio(m)
    mean_ms, std_ms = time_forward_batches(
        m, test_loader, device, max_batches=MAX_TIMING_BATCHES
    )
    rows.append(
        {
            "checkpoint": name,
            "test_acc": acc,
            "trainable_params": npar,
            "param_mib": pb / (1024**2),
            "weight_sparsity_ratio": sp,
            "latency_mean_ms": mean_ms,
            "latency_std_ms": std_ms,
            "file_mib": ckpt.stat().st_size / (1024**2),
        }
    )

display(pd.DataFrame(rows))


/Users/carlosvalerio/Documents/Study/Aprendizaje/pia/.venv/lib/python3.13/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


,checkpoint,test_acc,trainable_params,param_mib,weight_sparsity_ratio,latency_mean_ms,latency_std_ms,file_mib
0,baseline-test (model_final),0.4885,337770,1.288490,0.002031,0.332311,0.196760,1.290776
1,structured-pruning (model_final),0.5632,219586,0.837654,0.100152,0.194113,0.090735,0.839910


In [5]:
# Tras evaluar en GPU/MPS: gc + vaciar caché del dispositivo (libera memoria del runtime).
"""gc.collect y empty_cache en CUDA/MPS para liberar memoria del dispositivo."""

from __future__ import annotations

import gc

import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()
print("gc.collect y empty_cache (CUDA/MPS) ejecutados.")


gc.collect y empty_cache (CUDA/MPS) ejecutados.
